# Coding Challenges: Deel Senior Full Stack Engineer

This notebook contains practical exercises focused on the Deel tech stack: **React, TypeScript, Node.js (Express), and PostgreSQL**.

---

## 1. Frontend: React Hooks & TypeScript

### Challenge: Create a `useEmployeeFilter` Hook

**Scenario:** You need to filter a large list of employees by their status (`active`, `onboarding`, `terminated`) and search term.

**Requirements:**
- Typed interfaces for Employee and Status.
- Use `useMemo` for performance.
- Handle empty search terms.

In [ ]:
// Mock TypeScript Implementation (Conceptual)
/*
type EmployeeStatus = 'active' | 'onboarding' | 'terminated';

interface Employee {
  id: string;
  name: string;
  status: EmployeeStatus;
  country: string;
}

const useEmployeeFilter = (employees: Employee[], searchTerm: string, statusFilter: EmployeeStatus | 'all') => {
  return useMemo(() => {
    return employees.filter(emp => {
      const matchesSearch = emp.name.toLowerCase().includes(searchTerm.toLowerCase());
      const matchesStatus = statusFilter === 'all' || emp.status === statusFilter;
      return matchesSearch && matchesStatus;
    });
  }, [employees, searchTerm, statusFilter]);
};
*/

## 2. Backend: Express Middleware & Security

### Challenge: Implement a Rate Limiter Middleware

**Scenario:** Protect the sensitive `/api/payroll/execute` endpoint from abuse.

**Requirements:**
- Limit to 5 requests per minute per IP.
- Return a 429 status code if exceeded.

In [ ]:
import express, { Request, Response, NextFunction } from 'express';

const rateLimitStore: Record<string, { count: number, startTime: number }> = {};

const simpleRateLimiter = (req: Request, res: Response, next: NextFunction) => {
  const ip = req.ip;
  const now = Date.now();
  const windowMs = 60000; // 1 minute
  const maxRequests = 5;

  if (!rateLimitStore[ip]) {
    rateLimitStore[ip] = { count: 1, startTime: now };
    return next();
  }

  const data = rateLimitStore[ip];

  if (now - data.startTime > windowMs) {
    rateLimitStore[ip] = { count: 1, startTime: now };
    return next();
  }

  if (data.count >= maxRequests) {
    return res.status(429).json({ error: 'Too many requests. Please try again later.' });
  }

  data.count++;
  next();
};

## 3. Database: SQL & Schema Design

### Challenge: Write a query to find companies with unpaid invoices over $10,000.

**Schema Snippet:**
- `companies` (id, name)
- `invoices` (id, company_id, amount, status, currency)

In [ ]:
/*
SELECT 
    c.name, 
    SUM(i.amount) as total_unpaid
FROM 
    companies c
JOIN 
    invoices i ON c.id = i.company_id
WHERE 
    i.status = 'unpaid'
GROUP BY 
    c.id, c.name
HAVING 
    SUM(i.amount) > 10000
ORDER BY 
    total_unpaid DESC;
*/

## 4. System Design: Global Payroll Flow

### Diagramming the logic (Mermaid Example)

1. Client triggers Payroll.
2. System calculates gross-to-net based on country rules.
3. System checks for sufficient funds.
4. Payments are dispatched via Wise/Revolut API.
5. Payslips are generated and emailed.
6. Accounting records are updated.